# **Hybrid LSTM + RF dengan Hyperparameter Tuning**

Here’s the LSTM + Random Forest Hybrid Model with Hyperparameter Tuning using GridSearchCV for the best accuracy. 🚀

This approach combines deep learning (LSTM) to extract meaningful features from text and Random Forest (RF) to classify sentiments.


**Steps in the Code**

1.   Dataset balancing antara: Positive, Neutral & Nagative.
2.   Label Encoding, Text Processing, Tokenisasi.
3.   Split dataset: Data Training dan Data Test
4.   LSTM Model extracts feature embeddings.
5.   Extracted Features (LSTM last hidden state) are used as input to Random Forest.
6.   Hyperparameter Tuning for Random Forest using GridSearchCV.
7.   Evaluasi model & prediction example


**Import Libraries**

In [ ]:
import pandas as pd
import numpy as np
import re
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout, Bidirectional, Input
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score
from sklearn.utils import resample

**Load & Balance Dataset**

In [ ]:
# Load dataset
df = pd.read_csv('/content/sample_data/data_berlabel.csv', encoding='utf-8')
df = df[['polarity', 'text_akhir']].dropna()

In [ ]:
df.head()

,polarity,text_akhir
0,positive,pengalaman pakai byu sinyalnya kayak telkomsel...
1,negative,parah parah parahhh udah error kunjung aja bel...
2,negative,haduhh parah operator buka aplikasi banget ber...
3,negative,aplikasi buruk jaringan jam jaringan buruk apl...
4,negative,hilang sinyallemot parah sinyalnya errornyanye...


In [ ]:
df.shape

(10775, 2)

In [ ]:
# Balancing Number of Dataset (negative, neutral, positive)
# Separate sentiment classes
df_negative = df[df['polarity'] == 'negative']
df_neutral = df[df['polarity'] == 'neutral']
df_positive = df[df['polarity'] == 'positive']

# Oversample neutral and positive classes to match negative
df_neutral_oversampled = resample(df_neutral, replace=True, n_samples=len(df_negative), random_state=42)
df_positive_oversampled = resample(df_positive, replace=True, n_samples=len(df_negative), random_state=42)

# Combine to create a balanced dataset
df_balanced = pd.concat([df_negative, df_neutral_oversampled, df_positive_oversampled]) # Use this for oversampling
df_balanced = df_balanced.sample(frac=1, random_state=42).reset_index(drop=True)  # Shuffle dataset

print(df_balanced['polarity'].value_counts())  # Check balance

polarity
negative    3615
neutral     3615
positive    3615
Name: count, dtype: int64


**Encode Labels & Clean Text**

In [ ]:
# Label Encoding
label_map = {'negative': 0, 'neutral': 1, 'positive': 2}
df_balanced['label'] = df_balanced['polarity'].map(label_map)

# Label Encoding
#y = LabelEncoder().fit_transform(df_balanced['airline_sentiment'])

# Text Preprocessing
def clean_text(text):
    text = re.sub(r'http\S+|www\S+', '', text)  # Remove URLs
    text = re.sub(r'[^a-zA-Z\s]', '', text).lower().strip()  # Remove special characters & lowercase
    return text

df_balanced['text_akhir'] = df_balanced['text_akhir'].apply(clean_text)

**Tokenization & Padding**

In [ ]:
# Tokenization
tokenizer = Tokenizer(num_words=13386)  # Limit vocabulary size
tokenizer.fit_on_texts(df_balanced['text_akhir'])
sequences = tokenizer.texts_to_sequences(df_balanced['text_akhir'])
X = pad_sequences(sequences, maxlen=100)

# Labels
y = df_balanced['label'].values

**Train-Test Split**

In [ ]:
# Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

**Build & Train LSTM Model**

In [ ]:
from tensorflow.keras.layers import Input, Embedding, Bidirectional, LSTM, Dropout, Dense
from tensorflow.keras.models import Model
from tensorflow.keras.regularizers import l2  # Import l2 regularizer

# Define the model
input_layer = Input(shape=(100,))
embedding_layer = Embedding(input_dim=13386, output_dim=50)(input_layer)
lstm_layer = Bidirectional(LSTM(128, return_sequences=False, kernel_regularizer=l2(0.01)))(embedding_layer)
dropout_layer = Dropout(0.3)(lstm_layer)
dense_layer = Dense(64, activation='relu', kernel_regularizer=l2(0.01))(dropout_layer)
output_layer = Dense(3, activation='softmax')(dense_layer)

# Create the model
lstm_model = Model(inputs=input_layer, outputs=output_layer)

# Compile the model
lstm_model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# Summary of the model
lstm_model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)           │ (None, 100)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ embedding_1 (Embedding)              │ (None, 100, 50)             │         669,300 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ bidirectional_1 (Bidirectional)      │ (None, 256)                 │         183,296 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_1 (Dropout)                  │ (None, 256)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_2 (Dense)                      │ (None, 64)                  │          16,448 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_3 (Dense)                      │ (None, 3)                   │             195 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 869,239 (3.32 MB)

 Trainable params: 869,239 (3.32 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.regularizers import l2  # Import l2 regularizer

# Convert labels to one-hot encoding
y_train_oh = to_categorical(y_train, num_classes=3)
y_test_oh = to_categorical(y_test, num_classes=3)

# Define LSTM Model
input_layer = Input(shape=(100,))
embedding_layer = Embedding(input_dim=13386, output_dim=50)(input_layer)
lstm_layer = Bidirectional(LSTM(128, return_sequences=False, kernel_regularizer=l2(0.01)))(embedding_layer)
dropout_layer = Dropout(0.3)(lstm_layer)
dense_layer = Dense(64, activation='relu', kernel_regularizer=l2(0.01))(dropout_layer)
feature_output = Dense(32, activation='relu')(dense_layer)

# **Final Softmax Layer for Classification**
output_layer = Dense(3, activation='softmax')(feature_output)

# Corrected Model
lstm_model = Model(inputs=input_layer, outputs=output_layer)
lstm_model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

In [ ]:
# Train The Model
lstm_model.fit(X_train, y_train_oh, validation_split=0.2, epochs=10, batch_size=64)

Epoch 1/10
109/109 ━━━━━━━━━━━━━━━━━━━━ 56s 464ms/step - accuracy: 0.3701 - loss: 2.6736 - val_accuracy: 0.5703 - val_loss: 1.0064
Epoch 2/10
109/109 ━━━━━━━━━━━━━━━━━━━━ 82s 464ms/step - accuracy: 0.5979 - loss: 0.9098 - val_accuracy: 0.6331 - val_loss: 0.8095
Epoch 3/10
109/109 ━━━━━━━━━━━━━━━━━━━━ 75s 402ms/step - accuracy: 0.7343 - loss: 0.6710 - val_accuracy: 0.7512 - val_loss: 0.6301
Epoch 4/10
109/109 ━━━━━━━━━━━━━━━━━━━━ 44s 406ms/step - accuracy: 0.8227 - loss: 0.5123 - val_accuracy: 0.7811 - val_loss: 0.5955
Epoch 5/10
109/109 ━━━━━━━━━━━━━━━━━━━━ 81s 397ms/step - accuracy: 0.8811 - loss: 0.3887 - val_accuracy: 0.8024 - val_loss: 0.5819
Epoch 6/10
109/109 ━━━━━━━━━━━━━━━━━━━━ 44s 408ms/step - accuracy: 0.9260 - loss: 0.2912 - val_accuracy: 0.8226 - val_loss: 0.5503
Epoch 7/10
109/109 ━━━━━━━━━━━━━━━━━━━━ 82s 409ms/step - accuracy: 0.9546 - loss: 0.2368 - val_accuracy: 0.8059 - val_loss: 0.6347
Epoch 8/10
109/109 ━━━━━━━━━━━━━━━━━━━━ 81s 405ms/step - accuracy: 0.9505 - loss: 0

**Extract LSTM Features & Normalize for RF**

In [ ]:
# Extract Features
X_train_features = lstm_model.predict(X_train)
X_test_features = lstm_model.predict(X_test)

# Normalize features
scaler = StandardScaler()
X_train_features = scaler.fit_transform(X_train_features)
X_test_features = scaler.transform(X_test_features)

272/272 ━━━━━━━━━━━━━━━━━━━━ 21s 74ms/step
68/68 ━━━━━━━━━━━━━━━━━━━━ 6s 90ms/step


**Train & Tune Random Forest**

In [24]:
# Define hyperparameter grid
param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [10, 20, None],
    'min_samples_split': [2, 5]
}

# Train Random Forest with Grid Search
rf = RandomForestClassifier(random_state=42)
grid_search = GridSearchCV(rf, param_grid, cv=3, scoring='accuracy', verbose=2, n_jobs=-1)
grid_search.fit(X_train_features, y_train)

# Best Model
best_rf = grid_search.best_estimator_

# Evaluate
y_pred = best_rf.predict(X_test_features)
accuracy = accuracy_score(y_test, y_pred)
print(f'Hybrid LSTM + RF Accuracy: {accuracy:.4f}')

Fitting 3 folds for each of 12 candidates, totalling 36 fits
Hybrid LSTM + RF Accuracy: 0.8797


**Prediction Function**

In [25]:
def predict_sentiment(text, lstm_model, rf_model, tokenizer, scaler, max_length=100):
    """Preprocess input text and predict sentiment using Hybrid LSTM + RF."""
    text_clean = clean_text(text)
    sequence = tokenizer.texts_to_sequences([text_clean])
    padded_sequence = pad_sequences(sequence, maxlen=max_length)

    # Extract features using LSTM
    features = lstm_model.predict_on_batch(padded_sequence)

    # Normalize features before RF prediction
    features = scaler.transform(features)

    # Predict sentiment with RF using probabilities
    sentiment_probs = rf_model.predict_proba(features)[0]
    sentiment_label = np.argmax(sentiment_probs)  # Select highest probability class

    return {0: 'negative', 1: 'neutral', 2: 'positive'}[sentiment_label]

**Test Predictions**

In [26]:
examples = [
    "Saya suka produk ini.",
    "Saya suka beli disini! Service mereka luar biasa.",
    "aplikasi buruk."
]

for text in examples:
    sentiment = predict_sentiment(text, lstm_model, best_rf, tokenizer, scaler)
    print(f"Text: {text}\nPredicted Sentiment: {sentiment}\n")

Text: Saya suka produk ini.
Predicted Sentiment: positive

Text: Saya suka beli disini! Service mereka luar biasa.
Predicted Sentiment: positive

Text: aplikasi buruk.
Predicted Sentiment: negative

